# 08 — Validation externe (Quiver)

**Rôle :** vérifier notre table contre Quiver (couverture par an / déclarant). **Quiver n'entre jamais dans la table finale** — c'est une sanity-check, pas une source.

**Prérequis :** `QUIVER_API_TOKEN` dans l'environnement · notebook 07 exécuté.

**Cellule 1 — Imports, chemins, token.**

In [1]:
import os, json
from pathlib import Path
import requests, pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED    = PROJECT_ROOT / "data" / "processed"
EXT_QUIVER   = PROJECT_ROOT / "data" / "external" / "quiver"; EXT_QUIVER.mkdir(parents=True, exist_ok=True)
REPORTS      = PROJECT_ROOT / "reports"
TOKEN = os.environ.get("QUIVER_API_TOKEN", "").strip()
print("Token présent :", bool(TOKEN))

Token présent : True


**Cellule 2 — Client Quiver minimal** (dans le notebook, pas de dépendance opaque). Endpoint bulk congresstrading V2.

In [2]:
import time

BASE = "https://api.quiverquant.com"
ENDPOINT = "/beta/bulk/congresstrading"

def quiver_page(page, page_size=500, version="V2", retries=3):
    headers = {"Authorization": f"Bearer {TOKEN}"}
    params = {"version": version, "normalized": "true", "nonstock": "false",
              "page": page, "page_size": page_size}
    for attempt in range(retries):
        r = requests.get(BASE + ENDPOINT, headers=headers, params=params, timeout=60)
        if r.status_code == 429:
            wait = 60 * (attempt + 1)
            print(f"  429 page {page} — attente {wait}s (tentative {attempt+1}/{retries})")
            time.sleep(wait)
            continue
        r.raise_for_status()
        data = r.json()
        return data if isinstance(data, list) else data.get("results", data.get("data", []))
    raise RuntimeError(f"Page {page} : 429 persistant après {retries} tentatives")

**Cellule 3 — Récupération paginée** (réglez `MAX_PAGES`).

In [3]:
CACHE = EXT_QUIVER / "quiver_congress.json"
if CACHE.exists():
    recs = json.loads(CACHE.read_text())
    print(f"Cache existant : {len(recs)} records")
else:
    MAX_PAGES = 30  # 30 × 500 = 15 000 records, suffisant pour la validation
    recs = []
    for p in range(1, MAX_PAGES + 1):
        page = quiver_page(p)
        if not page:
            print(f"Page {p} vide — fin à {len(recs)} records")
            break
        recs.extend(page)
        print(f"  page {p} → +{len(page)} (total {len(recs)})", flush=True)
        time.sleep(1.5)  # ~40 req/min, sous la limite Quiver
    CACHE.write_text(json.dumps(recs))
    print(f"Records Quiver : {len(recs)}")

  page 1 → +500 (total 500)


  page 2 → +500 (total 1000)


  page 3 → +500 (total 1500)


  page 4 → +500 (total 2000)


  page 5 → +500 (total 2500)


...

  page 27 → +500 (total 13500)


  page 28 → +500 (total 14000)


  page 29 → +500 (total 14500)


  page 30 → +500 (total 15000)


Records Quiver : 15000


**Cellule 4 — Normalisation Quiver** (pour comparaison seulement).

In [4]:
q = pd.DataFrame(recs)
def g(df, *n):
    for x in n:
        if x in df.columns:
            return df[x]
    return pd.Series([None] * len(df))
quiver = pd.DataFrame({
    "declarant_name":   g(q, "Name", "Representative"),
    "chamber":          g(q, "Chamber", "House"),
    "ticker":           g(q, "Ticker"),
    "transaction_date": pd.to_datetime(g(q, "Traded", "TransactionDate"), errors="coerce"),
    "disclosure_date":  pd.to_datetime(g(q, "Filed", "ReportDate"), errors="coerce"),
})
print(len(quiver), "lignes Quiver")

15000 lignes Quiver


**Cellule 5 — Triangulation de couverture par an.** Un écart Quiver ≠ une erreur de notre côté.

In [5]:
ours = pd.read_csv(PROCESSED / "congress_transactions.csv", parse_dates=["disclosure_date"])
def by_year(df):
    return df["disclosure_date"].dt.year.value_counts().sort_index()
cmp = pd.DataFrame({"notre_table": by_year(ours), "quiver": by_year(quiver)}).fillna(0).astype(int)
cmp

,notre_table,quiver
disclosure_date,,
2014,1289,0
2015,2395,0
2016,2269,0
2017,2620,0
2018,1995,0
2019,1996,0
2020,3041,0
2021,2108,0
2022,2002,0


**Cellule 6 — Rapport de validation.**

In [6]:
from datetime import datetime, timezone
lines = ["# Validation Quiver (vérification externe)", "",
         f"Généré : {datetime.now(timezone.utc).isoformat(timespec='seconds')}", "",
         f"- Notre table : {len(ours)} transactions, {ours['declarant_name'].nunique()} déclarants",
         f"- Quiver (échantillon) : {len(quiver)} transactions, {quiver['declarant_name'].nunique()} déclarants", "",
         "## Couverture par an", "```", cmp.to_string(), "```", "",
         "> Quiver est une vérification ; jamais réinjecté dans la table finale."]
(REPORTS / "validation_report.md").write_text("\n".join(lines) + "\n")
print("Rapport écrit.")

Rapport écrit.


Validation faite ✅ — rapport final **`09_data_quality_report`**.